In [1]:
import pandas as pd
import re
import zipfile
import urllib.request
from io import BytesIO

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

# Télécharger le dataset SMS Spam Collection
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"

response = urllib.request.urlopen(url)
archive = zipfile.ZipFile(BytesIO(response.read()))

with archive.open("SMSSpamCollection") as file:
    df = pd.read_csv(
        file,
        sep="\t",
        header=None,
        names=["label", "text"],
        encoding="utf-8"
    )

print(df.head())
print(df["label"].value_counts())

  label                                               text
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...
label
ham     4825
spam     747
Name: count, dtype: int64


In [2]:
def nettoyer_texte(texte):
    texte = texte.lower()
    texte = re.sub(r"http\S+|www\.\S+", " ", texte)
    texte = re.sub(r"[^a-z\s]", " ", texte)
    texte = re.sub(r"\s+", " ", texte).strip()
    return texte

df["text_clean"] = df["text"].apply(nettoyer_texte)

print(df[["text", "text_clean"]].head())

                                                text  \
0  Go until jurong point, crazy.. Available only ...   
1                      Ok lar... Joking wif u oni...   
2  Free entry in 2 a wkly comp to win FA Cup fina...   
3  U dun say so early hor... U c already then say...   
4  Nah I don't think he goes to usf, he lives aro...   

                                          text_clean  
0  go until jurong point crazy available only in ...  
1                            ok lar joking wif u oni  
2  free entry in a wkly comp to win fa cup final ...  
3        u dun say so early hor u c already then say  
4  nah i don t think he goes to usf he lives arou...  


In [17]:
modeles = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Linear SVM": LinearSVC()
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

resultats = {}

for nom, modele in modeles.items():

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english")),
        ("classifier", modele)
    ])

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy"
    )

    resultats[nom] = scores.mean()

    print(nom)
    print("Scores des 5 folds :", scores)
    print("Accuracy moyenne :", round(scores.mean(), 4))
    print("-------------------------")

Naive Bayes
Scores des 5 folds : [0.97197309 0.96076233 0.9674523  0.96296296 0.96184063]
Accuracy moyenne : 0.965
-------------------------
Logistic Regression
Scores des 5 folds : [0.95964126 0.95067265 0.95735129 0.94612795 0.95173962]
Accuracy moyenne : 0.9531
-------------------------
Linear SVM
Scores des 5 folds : [0.98206278 0.97869955 0.97755331 0.98316498 0.98204265]
Accuracy moyenne : 0.9807
-------------------------


In [16]:
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [14]:
scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="accuracy"
)

In [12]:
X = df["text"]
y = df["label"]

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
evaluations = []

for nom, modele in modeles.items():

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english")),
        ("classifier", modele)
    ])

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)

    evaluations.append({
        "Modèle": nom,
        "Accuracy CV": resultats[nom],
        "Accuracy Test": accuracy
    })

    print("\nModèle :", nom)
    print(classification_report(
        y_test,
        predictions,
        zero_division=0
    ))

tableau = pd.DataFrame(evaluations)
print(tableau.sort_values("Accuracy Test", ascending=False))